# Entity Recognition System - Colab Free T4 Training

This notebook trains both models planned for the project:

1. A spaCy NER model
2. A Hugging Face transformer token-classification model

It is designed for Google Colab free tier. Use `Runtime > Change runtime type > T4 GPU` when available. If no GPU is available, the spaCy section still runs comfortably, and the transformer section can run more slowly on CPU.

References:
- Hugging Face token classification task guide: https://huggingface.co/docs/transformers/tasks/token_classification
- spaCy training guide: https://spacy.io/usage/training

## 1. Colab Runtime Check

Run this first to confirm whether Colab assigned a GPU.

In [ ]:
import os
import platform
import torch

print("Python:", platform.python_version())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU assigned. You can still run spaCy training; transformer training will be slower.")

## 2. Clone Project and Install Dependencies

This clones your GitHub repo and installs only the packages needed for training in Colab. `accelerate` is included because Hugging Face `Trainer` needs it in many current Colab environments.

In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/MuhammadJamall/ers.git"
PROJECT_DIR = "/content/ers"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "accelerate", "openpyxl"], check=True)

## 3. Optional: Mount Google Drive for Saved Models

Run this cell if you want trained models copied to Google Drive at the end.

In [ ]:
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ers-trained-models'
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print('Drive output:', DRIVE_OUTPUT_DIR)
else:
    DRIVE_OUTPUT_DIR = None
    print('Google Drive saving disabled.')

## 4. Validate the Dataset

The full training dataset is stored as `data/processed/ner_starter_bio.csv`.

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-m",
    "src.validate_dataset",
    "--path",
    "data/processed/ner_starter_bio.csv",
], check=True)

## 5. Load BIO Dataset

The dataset already includes train, validation, and test splits. We will keep those splits to avoid sentence leakage.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from src.config import LABELS, LABEL_TO_ID, ID_TO_LABEL

DATA_PATH = Path('data/processed/ner_starter_bio.csv')
df = pd.read_csv(DATA_PATH)

required_columns = {'split', 'sentence_id', 'token_id', 'token', 'label'}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f'Missing columns: {sorted(missing)}')

print(df.head())
print()
print('Splits:')
print(df['split'].value_counts())
print()
print('Labels:')
print(df['label'].value_counts().sort_index())

## 6. Convert BIO Rows to Sentence Records

Both spaCy and Hugging Face need sentence-level records. Each record contains a list of tokens and a list of labels.

In [ ]:
def build_sentence_records(dataframe: pd.DataFrame) -> list[dict]:
    records = []
    ordered = dataframe.sort_values(['split', 'sentence_id', 'token_id'])
    for (split, sentence_id), group in ordered.groupby(['split', 'sentence_id'], sort=False):
        tokens = group['token'].astype(str).tolist()
        labels = group['label'].astype(str).tolist()
        records.append({
            'split': split,
            'sentence_id': int(sentence_id),
            'tokens': tokens,
            'labels': labels,
            'ner_tags': [LABEL_TO_ID[label] for label in labels],
        })
    return records

records = build_sentence_records(df)
records_by_split = {
    split: [record for record in records if record['split'] == split]
    for split in ['train', 'validation', 'test']
}

for split, split_records in records_by_split.items():
    print(split, len(split_records), 'sentences')

records_by_split['train'][0]

# Part A - Hugging Face Transformer NER

This section fine-tunes `distilbert-base-uncased`, which is a good Colab-free choice because it is much lighter than BERT-large or RoBERTa-large.

## 7. Build Hugging Face Dataset

In [ ]:
from datasets import Dataset, DatasetDict

hf_dataset = DatasetDict({
    split: Dataset.from_list(split_records)
    for split, split_records in records_by_split.items()
})

hf_dataset

## 8. Tokenize and Align Labels

Transformer tokenizers may split one word into multiple subwords. We label only the first subword and set the rest to `-100`, which PyTorch ignores in the loss.

In [ ]:
from transformers import AutoTokenizer

MODEL_CHECKPOINT = 'distilbert-base-uncased'
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,
    )

    aligned_labels = []
    for batch_index, labels in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)
            previous_word_id = word_id

        aligned_labels.append(label_ids)

    tokenized_inputs['labels'] = aligned_labels
    return tokenized_inputs

tokenized_dataset = hf_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_dataset

## 9. Metrics

In [ ]:
import evaluate

seqeval = evaluate.load('seqeval')
label_list = LABELS

def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[pred] for pred, label in zip(prediction, label_row) if label != -100]
        for prediction, label_row in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[label] for pred, label in zip(prediction, label_row) if label != -100]
        for prediction, label_row in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }

## 10. Train Transformer Model

Free Colab tips:

- Keep `distilbert-base-uncased` for the first run.
- Use batch size 8 on T4 if memory is tight.
- Use 3 epochs first; increase only if you have GPU time left.
- Keep checkpoint saves limited to avoid filling the Colab disk.

In [ ]:
import inspect
import torch
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

TRANSFORMER_OUTPUT_DIR = 'models/transformer/distilbert-ner'

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_kwargs = {
    'output_dir': TRANSFORMER_OUTPUT_DIR,
    'learning_rate': 2e-5,
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 8,
    'gradient_accumulation_steps': 2,
    'num_train_epochs': 3,
    'weight_decay': 0.01,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'logging_steps': 20,
    'save_total_limit': 2,
    'report_to': 'none',
    'fp16': torch.cuda.is_available(),
}

# Transformers v5 uses eval_strategy; many v4 installs use evaluation_strategy.
training_arg_names = inspect.signature(TrainingArguments.__init__).parameters
if 'eval_strategy' in training_arg_names:
    training_kwargs['eval_strategy'] = 'epoch'
else:
    training_kwargs['evaluation_strategy'] = 'epoch'

training_args = TrainingArguments(**training_kwargs)

trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': tokenized_dataset['train'],
    'eval_dataset': tokenized_dataset['validation'],
    'data_collator': data_collator,
    'compute_metrics': compute_metrics,
}

# Transformers v5 renamed tokenizer to processing_class.
trainer_arg_names = inspect.signature(Trainer.__init__).parameters
if 'processing_class' in trainer_arg_names:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)
trainer.train()

## 11. Evaluate and Save Transformer Model

In [ ]:
validation_metrics = trainer.evaluate(tokenized_dataset['validation'])
test_metrics = trainer.evaluate(tokenized_dataset['test'])

print('Validation metrics:', validation_metrics)
print('Test metrics:', test_metrics)

trainer.save_model(TRANSFORMER_OUTPUT_DIR)
tokenizer.save_pretrained(TRANSFORMER_OUTPUT_DIR)

Path('reports').mkdir(exist_ok=True)
with open('reports/transformer_validation_metrics.json', 'w', encoding='utf-8') as file:
    json.dump(validation_metrics, file, indent=2)
with open('reports/transformer_test_metrics.json', 'w', encoding='utf-8') as file:
    json.dump(test_metrics, file, indent=2)

## 12. Transformer Inference Test

In [ ]:
from transformers import pipeline

ner_pipeline = pipeline(
    'ner',
    model=TRANSFORMER_OUTPUT_DIR,
    tokenizer=TRANSFORMER_OUTPUT_DIR,
    aggregation_strategy='simple',
    device=0 if torch.cuda.is_available() else -1,
)

sample_text = 'Ali Raza joined Systems Limited in Lahore on 15 June 2026 to work on machine learning.'
ner_pipeline(sample_text)

# Part B - spaCy NER

This section trains a lightweight spaCy NER model from the same BIO dataset.

## 13. Convert BIO Records to spaCy DocBin

In [ ]:
import spacy
from spacy.tokens import Doc, DocBin

SPACY_DATA_DIR = Path('data/processed/spacy')
SPACY_DATA_DIR.mkdir(parents=True, exist_ok=True)

def bio_labels_to_spans(doc: Doc, labels: list[str]):
    spans = []
    start = None
    current_entity = None

    for index, label in enumerate(labels):
        if label == 'O':
            if start is not None:
                spans.append(doc[start:index])
                start = None
                current_entity = None
            continue

        prefix, entity_type = label.split('-', 1)
        if prefix == 'B' or entity_type != current_entity:
            if start is not None:
                spans.append(doc[start:index])
            start = index
            current_entity = entity_type

    if start is not None:
        spans.append(doc[start:len(labels)])

    for span in spans:
        span.label_ = span.label_

    return spans

def make_spacy_docbin(split_records: list[dict], output_path: Path) -> None:
    nlp = spacy.blank('en')
    doc_bin = DocBin()

    for record in split_records:
        words = record['tokens']
        spaces = [True] * len(words)
        if spaces:
            spaces[-1] = False
        doc = Doc(nlp.vocab, words=words, spaces=spaces)

        spans = []
        start = None
        entity_type = None
        labels = record['labels']

        for index, label in enumerate(labels):
            if label == 'O':
                if start is not None:
                    spans.append(doc.char_span(doc[start].idx, doc[index - 1].idx + len(doc[index - 1]), label=entity_type))
                    start = None
                    entity_type = None
                continue

            prefix, current_type = label.split('-', 1)
            if prefix == 'B' or current_type != entity_type:
                if start is not None:
                    spans.append(doc.char_span(doc[start].idx, doc[index - 1].idx + len(doc[index - 1]), label=entity_type))
                start = index
                entity_type = current_type

        if start is not None:
            spans.append(doc.char_span(doc[start].idx, doc[len(doc) - 1].idx + len(doc[len(doc) - 1]), label=entity_type))

        doc.ents = [span for span in spans if span is not None]
        doc_bin.add(doc)

    doc_bin.to_disk(output_path)

make_spacy_docbin(records_by_split['train'], SPACY_DATA_DIR / 'train.spacy')
make_spacy_docbin(records_by_split['validation'], SPACY_DATA_DIR / 'validation.spacy')
make_spacy_docbin(records_by_split['test'], SPACY_DATA_DIR / 'test.spacy')

print('Saved spaCy DocBin files to', SPACY_DATA_DIR)

## 14. Create spaCy Config

In [ ]:
import subprocess
import sys

Path('configs').mkdir(exist_ok=True)
subprocess.run([
    sys.executable,
    "-m",
    "spacy",
    "init",
    "config",
    "configs/spacy_ner_config.cfg",
    "--lang",
    "en",
    "--pipeline",
    "ner",
    "--optimize",
    "efficiency",
    "--force",
], check=True)

## 15. Train spaCy Model

This model is lightweight and should run on free Colab. If the GPU is available, spaCy can use it; if not, CPU is fine for this dataset size.

In [ ]:
SPACY_OUTPUT_DIR = 'models/spacy/ner_model'

spacy_train_command = [
    sys.executable,
    "-m",
    "spacy",
    "train",
    "configs/spacy_ner_config.cfg",
    "--output",
    SPACY_OUTPUT_DIR,
    "--paths.train",
    "data/processed/spacy/train.spacy",
    "--paths.dev",
    "data/processed/spacy/validation.spacy",
]

if torch.cuda.is_available():
    spacy_train_command.extend(["--gpu-id", "0"])

subprocess.run(spacy_train_command, check=True)

## 16. Evaluate spaCy Model

In [ ]:
subprocess.run([
    sys.executable,
    "-m",
    "spacy",
    "evaluate",
    "models/spacy/ner_model/model-best",
    "data/processed/spacy/test.spacy",
    "--output",
    "reports/spacy_test_metrics.json",
], check=True)

## 17. spaCy Inference Test

In [ ]:
nlp = spacy.load('models/spacy/ner_model/model-best')
doc = nlp('Ali Raza joined Systems Limited in Lahore on 15 June 2026 to work on machine learning.')
[(ent.text, ent.label_) for ent in doc.ents]

## 18. Save Artifacts to Google Drive

Run this after training if `USE_GOOGLE_DRIVE = True` near the top of the notebook.

In [ ]:
import shutil

if DRIVE_OUTPUT_DIR:
    drive_output = Path(DRIVE_OUTPUT_DIR)
    drive_output.mkdir(parents=True, exist_ok=True)

    targets = [
        (Path('models/transformer/distilbert-ner'), drive_output / 'transformer_distilbert_ner'),
        (Path('models/spacy/ner_model'), drive_output / 'spacy_ner_model'),
        (Path('reports'), drive_output / 'reports'),
    ]

    for source, destination in targets:
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(source, destination)

    print('Saved artifacts to:', DRIVE_OUTPUT_DIR)
else:
    print('Drive saving skipped. Set USE_GOOGLE_DRIVE = True and rerun the Drive cells if needed.')

## 19. Suggested Free Colab Settings

If training disconnects or GPU time is short:

- Start with the transformer model first because it benefits most from T4.
- Keep `num_train_epochs = 3` for the first run.
- Keep `per_device_train_batch_size = 8` and `gradient_accumulation_steps = 2`.
- If CUDA memory runs out, reduce `per_device_train_batch_size` to 4.
- Save to Google Drive after training so Colab cleanup does not remove your models.